# Hidden states of the frozen T5-base encoder at scale: the task-recognition rules on two benchmarks and on 120 tasks

Nothing is trained. Per task, the frozen T5-base encoder (fp32, eval, no adapter) turns each instance text (the **input only**, the task
definition is never read; 256 tokens) into one vector (mean of the last-layer states over the tokens), standardised with the mean and
standard deviation of a pool of 2,000 unrelated inputs (embedded below, computed on a CPU beforehand). Three rules then give every test input
a task after each arrival: **L** (ridge, lam 1), **H'** (joint ridge with a handicap, t = 2, floor) and **J** (joint logistic regression, C = 1;
only for sequences of at most 30 tasks). The measures are those of `paper_certified/hidden_rules_study.py` (the code is copied from it).

Sequences: SuperNI order 1 and 2 (a check of the code against the CPU numbers), Long Sequence order 3 and 4 (all official test inputs),
fresh30 / fresh60 / fresh90 (90 tasks of the natural-instructions repository, batches 1-3, downloaded by name at a fixed commit),
and all120 = SuperNI order 1 + Long Sequence order 3 + the 90 fresh tasks.

**On Kaggle:** Accelerator GPU T4, Internet on, then *Save & Run All*. About 20-40 minutes (the encoding is about 10-15 minutes on the GPU; the
rule J is the slow part). Result: `/kaggle/working/hidden_states_scale/results.json` (written after every step; also printed as tables).

## Settings and imports

In [ ]:
QUICK = False        # True: every task restricted to 40 training and 20 test inputs, t5-small (a test of the logic on a CPU); the real run is False
import os, sys, json, math, random, time, subprocess, hashlib, re, shutil
import numpy as np
from scipy.stats import beta as beta_dist
from sklearn.linear_model import LogisticRegression

OUT_DIR = os.environ.get("HS_OUT", "/kaggle/working/hidden_states_scale")
_tmp = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp"
WORK = os.environ.get("HS_WORK", os.path.join(_tmp, "hidden_states_scale"))      # clone, downloaded inputs, feature cache (not part of the output)
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(WORK, exist_ok=True)
OFFICIAL_URL = "https://github.com/liangyanshuo/gainlora"
OFFICIAL_COMMIT = "e9d0d700538c8d12892f7fd52b216291f7e42acc"
NI_COMMIT = "55a365637381ce7f3748fa2eac7aef1a113bbb82"
NI_RAW = f"https://raw.githubusercontent.com/allenai/natural-instructions/{NI_COMMIT}/"
MAXTOK = 256
T0 = time.time()
def log(*a): print(f"[{time.time() - T0:6.0f}s]", *a, flush=True)
log("QUICK =", QUICK, "| output", OUT_DIR, "| work", WORK)

## Embedded constants
The 90 fresh tasks (name, number of parsed instances, sha1 fingerprint of the first 1000 inputs joined with the character 0x1e, computed on the
CPU files) and the mean / standard deviation (float32 values, written with full precision) of the 2,000-input pool, from `pool_t5hidden.npy`
(sha1 of the array bytes below) exactly as `standardiser()` computes them (std below 1e-6 set to 1).

In [ ]:
CONST = {'fresh': [['task469_mrqa_answer_generation', 1000, '124c984476755e69b0df9b3070365ace85ed420f'], ['task170_hotpotqa_answer_generation', 1000, 'cd64b3f60b1c81362ea5b3df0a4bc964eacf0105'], ['task1212_atomic_classification_hasproperty', 1000, 'ccf446a95a236274b3f60837722bb2bf3e5d523b'], ['task159_check_frequency_of_words_in_sentence_pair', 1000, 'd7a9fee75c3fd7196e1901478e038c7666353170'], ['task1284_hrngo_informativeness_classification', 1000, 'bc9c1bd7366659b2841c84b0bed93e12e6e2869a'], ['task1533_daily_dialog_formal_classification', 1000, 'cda32eacf1ea2f7c10081eab306b024a669aad13'], ['task1422_mathqa_physics', 1000, '74b8f8b8cdcf0693c123ea96fb318dbbdeb565ec'], ['task955_wiki_auto_style_transfer', 1000, '7d904b879bd6dfc67a0ca1287a526d68f7a5e5e2'], ['task343_winomt_classification_profession_anti', 1000, 'e0c3fa97a7d6026a631513cd2691a43f03fed18e'], ['task184_break_generate_question', 1000, 'a39189f8166835247874eaf32a3f93635189a19c'], ['task684_online_privacy_policy_text_information_type_generation', 1000, 'd2d9587e0fe3420f95bf8a29e046fddff153c06d'], ['task936_defeasible_nli_snli_classification', 1000, 'a959c0ad848f06ce26d70aa4ac683963c6f09c09'], ['task287_casehold_legal_incorrect_answer_generation', 1000, '2ec6b7f36cac06a16d7f0da659fcd753efbf972d'], ['task1631_openpi_answer_generation', 1000, 'de8b8c36540531f65098cb94e4787e767c6b460f'], ['task383_matres_classification', 1000, '04c4499b7bc6dd7af7b829c21f973d3044d84b84'], ['task521_trivia_question_classification', 1000, 'fb01235ae697b334ecb67251b89bd8adc9582017'], ['task028_drop_answer_generation', 1000, 'b21fafb1176ce22087f36a332f9cb78a293c1d03'], ['task1542_every_ith_element_from_starting', 1000, 'aacaee8c77c349b95c98eb2a23be06ab5bec7a79'], ['task418_persent_title_generation', 1000, '1e6e46f9e54b30910e6ffa0fa995ee9bb0adb35a'], ['task576_curiosity_dialogs_answer_generation', 1000, '75ece3fa285d593738172b9fc2e96ff699c46bea'], ['task374_synthetic_pos_or_neg_calculation', 1000, 'a97bd34fef97a3a53395542e4c56fa6bfa614733'], ['task323_jigsaw_classification_sexually_explicit', 1000, '7447522ab589fe4c7d82f5e9f54af246f4f9e672'], ['task330_gap_answer_generation', 1000, 'd4d7f246fa4ad92d4e185ff59ee7a42e8eb4a7da'], ['task629_dbpedia_14_classification', 1000, 'cf53e58d934e838d13bc2da28629b74740588d84'], ['task090_equation_learner_algebra', 1000, '27ca072e018b117937b86751351813b12b97153c'], ['task515_senteval_odd_word_out', 1000, 'dd939fa4f4553910471d4d12d371edffec5de07d'], ['task649_race_blank_question_generation', 1000, 'df46b90e4297692b8f5600d167c2e8d2771bfbc3'], ['task1594_yahoo_answers_topics_question_generation', 1000, '97358158435b87e7029234a10ade1526580b588d'], ['task1296_wiki_hop_question_answering', 1000, '84d93165d53d1e135d53511e5a70b07ed3bd105a'], ['task1288_glue_mrpc_paraphrasing', 1000, '43c86c36c226b28fc7d8239b6ceee03ee95b6570'], ['task1480_gene_extraction_jnlpba_dataset', 1000, '9e1de1787550404876e28c325d3a0a414c7f18fe'], ['task108_contextualabusedetection_classification', 1000, 'bf1f2620a10f417f892feffe6729f4dbf699d2c5'], ['task1727_wiqa_what_is_the_effect', 1000, '8e1e6c135f11a937401e895498cd11b0149833bb'], ['task1661_super_glue_classification', 1000, '5cae2b1b14f39b17d1971bfaea803cf79f1db28c'], ['task124_conala_pair_averages', 1000, '272cbe3a3b03b51b533e7415663216c86ef42cb9'], ['task1495_adverse_drug_event_classification', 1000, '35a7a2fc619eb8f647c94d002a3fde80be43316e'], ['task200_mnli_entailment_classification', 1000, '59b0c1b12a98a56608bf284333a7b2c90d94ca27'], ['task956_leetcode_420_strong_password_check', 1000, '6362bffda99d9fe38f70a8d1a984ebcb223c6669'], ['task372_synthetic_palindrome_numbers', 1000, '60ebcbb09af2b2d3542b26bd2d90b963090fcfe7'], ['task209_stancedetection_classification', 1000, '9a3cdc844884c0d0e732959462871c76ad1718f5'], ['task074_squad1.1_question_generation', 1000, 'd211c14e81ebd45c39f704bd11d4e8faf7a33d95'], ['task497_extract_all_numbers_from_list_in_order', 1000, '448baa3b6ac1db5781a3dc1023a9a0d9fa9696c8'], ['task357_casino_classification_negotiation_small_talk', 1000, '064ea53fd0e5d9d8a96981060488fd063e9c1393'], ['task1214_atomic_classification_xwant', 1000, '2211060d242148a5edabd91302244b5ae73a3e3a'], ['task584_udeps_eng_fine_pos_tagging', 1000, '4c5d9621e3279f039db335695d916c177f5608c6'], ['task223_quartz_explanation_generation', 1000, '9c31ad7c828138738bfe28985836169490ee50d2'], ['task934_turk_simplification', 1000, '8cbe426a89768ba834d0ea5d46a993f45a4512b6'], ['task102_commongen_sentence_generation', 1000, '31ad24a7e691692f521b365bccb0d7ac8b31a1be'], ['task306_jeopardy_answer_generation_double', 1000, '64ee7d8eee407b80408a4be35c5e196fd961b1a4'], ['task594_sciq_question_generation', 1000, '814e45064fe752c8438dc34bd054ed0d39e910cf'], ['task228_arc_answer_generation_easy', 1000, '3afdb76c953ef33e4b41fd820cfa411f67ac595c'], ['task304_numeric_fused_head_resolution', 1000, 'bbe31f9cfd0fd299465e91eee6ad16b9f3c16ed5'], ['task430_senteval_subject_count', 1000, 'b9cd25a75dba9df7407f6c16c176417787254908'], ['task402_grailqa_paraphrase_generation', 1000, 'f04d81079a368da45612676de531b7d46c3fa85a'], ['task367_synthetic_remove_floats', 1000, '57ccbb3bc6379e04e0a5ea3c08aac33f3640dc6b'], ['task455_swag_context_generation', 1000, 'c9bdb33eb20b3dcd48a019dcb94e92ade8cf64d3'], ['task1325_qa_zre_question_generation_on_subject_relation', 1000, 'c4f06ea04a3b00146129e1d6e763cccd6933a6ed'], ['task1507_boolean_temporal_reasoning', 1000, 'ce32361a3f8427af08cd375cd0a943b1d6fcd5ec'], ['task631_dbpedia_14_incorrect_answer_generation', 1000, '6b757a33558882589dcd66c71589db28cd3a0c41'], ['task210_logic2text_structured_text_generation', 1000, '4d5369a426db7a4338d92de899564917b7dc4583'], ['task075_squad1.1_answer_generation', 1000, '2285f890746036b0bda9496600baacb938a09985'], ['task593_sciq_explanation_generation', 1000, 'f64cbabe49cef42269696d1e22bb5c324109a67a'], ['task517_emo_classify_emotion_of_dialogue', 1000, 'e6869d6b76ab3182f8c56c27e153977ec4faec31'], ['task1286_openbookqa_question_answering', 1000, '27db6b5d92466e833883b26683bbe837a5699837'], ['task317_crows-pairs_classification_stereotype_type', 1000, '0a1b1571e238cc1f066bad727ec48730c6de5ea4'], ['task245_check_presence_in_set_intersection', 1000, 'dc41d85f1a47b000afb2612653bcc6239f514d5a'], ['task400_paws_paraphrase_classification', 1000, '644bb38d4c8e29679b26d8bf648d234eb9095c01'], ['task280_stereoset_classification_stereotype_type', 1000, '7fb569a39940277a558469fac86ec4643193bb04'], ['task567_circa_text_generation', 1000, 'ffcb775d56668b44ea840561dfb3e36aaffe574a'], ['task1391_winogrande_easy_answer_generation', 1000, '8b589d706a3f4c1d2599fa2df92cb76bf0a5396b'], ['task1731_quartz_question_answering', 1000, '4d523c16c93751c0548a9d6ea6db728675fcac9a'], ['task1660_super_glue_question_generation', 1000, 'e56f860e43fdc23483038a24dd9ad4ea75f14c70'], ['task1344_glue_entailment_classification', 1000, '0ee4e006723a36bf47929a9aa05ad187921e08ed'], ['task089_swap_words_verification', 1000, '66466b993d357f384c9f41594d68d5bbf86307ae'], ['task565_circa_answer_generation', 1000, '0c2dc37094e87ade671a844ce8174ba28a4d559a'], ['task391_causal_relationship', 1000, 'aa8370460034c6aafad10e6bfb55af09b1b759e8'], ['task574_air_dialogue_sentence_generation', 1000, '63c06711db5ee22324c07da6cfdad33c8441c9de'], ['task1615_sick_tclassify_b_relation_a', 1000, '6eec116f1fe37e96d4dbaf1571671502541bafe0'], ['task335_hateeval_classification_aggresive_en', 1000, '4e4182b4e0e89360b83412c238f4fd1e1ef7a611'], ['task095_conala_max_absolute_value', 1000, '145ea86e988e6cd7179c034fb9b4c69d0b4f7fe9'], ['task339_record_answer_generation', 1000, 'ac9e198f7d802003c6e5e0d2c581ef331daab5a2'], ['task592_sciq_incorrect_answer_generation', 1000, '2833e067f28e82b70099fbc8c5a2e2e8b18ea99a'], ['task399_semeval_2018_task1_tweet_sadness_detection', 1000, 'de001e25ca6ddb1d98190ed81ef057c55f676e05'], ['task327_jigsaw_classification_toxic', 1000, '75cab4247110a0ffb98435da294868c83410fab5'], ['task167_strategyqa_question_generation', 1000, 'cff8fa366fa2a3508cb58f99d9811334397643a0'], ['task738_perspectrum_classification', 1000, '05946ce6cb901dc330b9119316d410fab42eaf0a'], ['task377_remove_words_of_given_length', 1000, '629c2c456733ed05fad87a88eb3356c5f2a35960'], ['task564_discofuse_classification', 1000, '507e30637a509c5092579873f4a3292e8b98bb60'], ['task462_qasper_classification', 1000, '4aa204324ff89151843e6516550c320b02e75cf8'], ['task850_synthetic_longest_palindrome', 1000, 'd3af33842c13224491f05d69c380a6a069407672']], 'pool_sha1': '86afd1e85d51cf6733db68381d70fea63a496742', 'pool_shape': [2000, 768], 'mu': [-0.148545041680336, 0.038082439452409744, -0.06638997793197632, -0.01407154742628336, -0.007176019251346588, 0.05771031603217125, 0.0012990047689527273, -0.031122053042054176, -0.2080061137676239, -0.0487230159342289, -0.07182732969522476, 0.09817450493574142, -0.10662998259067535, 0.033230267465114594, -0.02195126563310623, 0.05049021914601326, -0.03245614096522331, 0.04730615392327309, 0.08825613558292389, -0.005051982123404741, -0.021209638565778732, -0.08474583923816681, 0.19521532952785492, 0.037450045347213745, -0.07055775821208954, 0.04493406042456627, 0.022180670872330666, 0.03523515537381172, -0.011060039512813091, -0.008879764005541801, 0.028291743248701096, 0.007739230524748564, 0.03520429506897926, -0.02622036449611187, 0.024554653093218803, 0.00394743587821722, -0.1354249119758606, -0.10327484458684921, 0.023336106911301613, 0.09434907138347626, -0.023737099021673203, -0.034699697047472, 0.08019600063562393, 0.011872896924614906, -0.47248706221580505, -0.02643468789756298, -0.029198337346315384, 0.028002720326185226, 0.028330568224191666, -0.0929294228553772, 0.10176615417003632, -0.03418789058923721, -0.1259862333536148, -0.04849998652935028, -0.3544240891933441, 0.027211224660277367, 0.39608532190322876, -0.013389344327151775, -0.013411987572908401, -0.024623030796647072, -0.05804905295372009, -0.0026310759130865335, -0.0066530839540064335, 0.022180890664458275, 0.02435225434601307, -0.14238005876541138, -0.0163654126226902, -0.014258976094424725, 0.028905119746923447, 0.014665840193629265, 0.022662585601210594, -0.03083132579922676, -0.016317784786224365, -0.02434941753745079, -0.026118353009223938, -0.006937631405889988, -0.03153437003493309, -0.014071102254092693, -0.05207438766956329, 0.04795027896761894, -0.009455306455492973, 0.04557066038250923, 0.06399080157279968, 0.059181299060583115, 0.034526754170656204, 0.06071047857403755, -0.022549165412783623, -0.01714867725968361, -0.03340204805135727, -0.016199013218283653, -0.014828121289610863, 0.03569392114877701, -0.002897632075473666, -0.0848812609910965, -0.008857020176947117, -0.01480246614664793, -0.00711361737921834, -0.015855874866247177, 0.029027622193098068, 0.004092687740921974, 0.03392625227570534, -0.019842181354761124, 0.11749841272830963, 0.0496678426861763, -0.09150249511003494, 0.10012661665678024, -0.08356982469558716, 0.05052315816283226, -0.07135111838579178, 0.15533024072647095, 0.0320105105638504, -0.031317196786403656, 0.02902526967227459, -0.0357801616191864, 0.021149447187781334, 0.04746381193399429, 0.03169575706124306, -0.06949660181999207, -0.0932733491063118, 0.0005936717498116195, -0.017255717888474464, 0.02311405912041664, -0.046040501445531845, -0.0006704818806611001, 0.05952424183487892, 0.013871214352548122, -0.04650692641735077, -0.03001396544277668, -0.046234726905822754, -0.040146809071302414, -0.10027316957712173, 0.06214580684900284, 0.00551755353808403, -0.0023266272619366646, -0.016040705144405365, -0.1632368415594101, -0.010749127715826035, 0.055529505014419556, -0.014367513358592987, -0.05713588371872902, 0.0436081700026989, 0.03374828025698662, 0.04049301892518997, 0.01367158256471157, 0.012363922782242298, 0.0015499534783884883, -0.044612158089876175, 0.025667423382401466, 0.020284296944737434, 0.001181783969514072, -0.0024513700045645237, -0.01048486027866602, -0.06682726740837097, 0.0029387008398771286, 0.0043893675319850445, 0.017078852280974388, 0.025293972343206406, -0.0315774641931057, -0.06431066244840622, -0.002247390104457736, 0.030762318521738052, 0.0307975672185421, 0.1085207462310791, 0.4373212456703186, -0.0116190817207098, -0.012129178270697594, 0.005921794567257166, -0.03821501508355141, 0.0010288949124515057, 0.37520912289619446, 0.06528313457965851, -0.015427119098603725, -0.033005114644765854, -0.0023397712502628565, -0.0017141202697530389, -0.016540607437491417, -0.04097965359687805, -0.007487773895263672, -0.04162231832742691, 0.19417214393615723, -0.01010865904390812, -0.036890819668769836, 0.050388921052217484, 0.03961896151304245, 0.018171751871705055, -0.017540544271469116, 0.010751795023679733, -0.017220867797732353, 0.12064613401889801, -0.08735612779855728, 0.028188876807689667, -0.012333493679761887, -0.010892723686993122, -0.003149573225528002, -0.03945819288492203, 0.15911394357681274, 0.020334254950284958, 0.05639994889497757, 0.06022921949625015, -0.041961848735809326, 0.02975885011255741, -0.0714883878827095, 0.053618185222148895, 0.03722171485424042, -0.014890620484948158, 0.02120441570878029, -0.004966405685991049, -0.0828721895813942, -0.0026739153545349836, 0.06385684013366699, 0.0322372205555439, -0.01585034281015396, -0.01598229445517063, -0.06583107262849808, -0.001815775060094893, -0.01519535481929779, -0.03571541607379913, -0.008649248629808426, 0.011305485852062702, -0.26807957887649536, -0.09765923768281937, -0.030095774680376053, -0.045200493186712265, -0.022734584286808968, 0.0008083238499239087, 0.013164117000997066, 0.022900095209479332, 0.0614955872297287, 0.02262292429804802, -0.03615708649158478, 0.026461727917194366, 0.033866770565509796, -0.04500335082411766, 0.03643251582980156, 0.021580325439572334, -0.04961617290973663, -0.18761558830738068, -0.08091539889574051, -0.012833314947783947, -0.07712741941213608, -0.09446196258068085, -0.14017142355442047, -0.002439662581309676, 0.01313546672463417, 0.004879518412053585, 0.0007258062832988799, 0.016266077756881714, -0.02064342238008976, 0.0008545765886083245, -0.10241483151912689, 0.3328877389431, -0.0008219809969887137, 0.0012804658617824316, 0.00860673375427723, -0.045197177678346634, 0.014363594353199005, 0.011981560848653316, -0.007953750900924206, 0.11413668841123581, 0.002615064149722457, -0.015901336446404457, 0.45157909393310547, -0.009492995217442513, -0.06236223876476288, -0.0554422065615654, 0.05082538351416588, 0.04684184491634369, 0.02294967696070671, 0.07321295887231827, 0.005399804096668959, -0.03963954746723175, -0.020208975300192833, 0.025790683925151825, -0.17967131733894348, -0.07914958894252777, -0.025295661762356758, -0.05289258062839508, -0.17100022733211517, 0.09977886080741882, 0.02854839898645878, 0.03715601563453674, 0.0667155310511589, 0.037386491894721985, 0.060301557183265686, -0.05384868383407593, -0.05561623349785805, 0.021888598799705505, -0.03917403519153595, -0.06326251477003098, 0.2880338132381439, 0.01427291240543127, -0.014164779335260391, -0.018085859715938568, -0.03085109032690525, 0.013601131737232208, 0.03444068878889084, -0.004411639180034399, 0.0024711608421057463, -0.06994990259408951, 0.017521247267723083, -0.06297583878040314, -0.006443777121603489, 0.06081034243106842, -0.035959381610155106, 0.011022595688700676, -0.040139567106962204, 0.02201489917933941, 0.018268760293722153, -0.008360518142580986, 0.024266837164759636, -0.01950603723526001, -0.07836664468050003, 0.0453350655734539, -0.08298806846141815, -0.0033835815265774727, 0.025516213849186897, -0.013773485086858273, 0.007688774261623621, 0.03775754198431969, 0.006521106231957674, 0.08037948608398438, 0.06859461963176727, -0.06382151693105698, 0.017226938158273697, 0.03832601010799408, -0.05303185433149338, 0.15410149097442627, -0.04807040095329285, 0.03568793088197708, -0.01669967547059059, -0.015844333916902542, 0.01647401973605156, 0.012507352977991104, -0.007565757259726524, 0.0017512822523713112, -0.0354246124625206, -0.021450109779834747, -0.002453131601214409, 0.03766016289591789, -0.0012504291953518987, 0.05852830037474632, -0.037131816148757935, 0.02377030998468399, 0.005975042004138231, -0.039410438388586044, 0.047827232629060745, -0.14108119904994965, 0.02613646723330021, 0.07655075192451477, -0.2717388868331909, -0.05356612429022789, -0.005169113632291555, -0.02670658566057682, -0.003311854787170887, -0.018291711807250977, -0.03722408786416054, -0.03798061981797218, -0.006119293626397848, -0.00894272979348898, 0.041711073368787766, -0.04329242184758186, -0.06897112727165222, -0.10683923959732056, 0.09256642311811447, 0.012840594165027142, -0.035383440554142, 0.08991488069295883, 0.034539468586444855, 0.03535594418644905, 0.010393030010163784, -0.059123169630765915, -0.11097913235425949, 0.06969714909791946, 0.05092480033636093, -0.08362864702939987, -0.012768873013556004, -0.08924129605293274, 0.05636432021856308, 0.07921944558620453, -0.1260530799627304, -0.01008193101733923, -0.009992880746722221, 0.045667704194784164, -0.023710867390036583, 0.016795968636870384, 0.040467679500579834, 0.021220151335000992, -0.09840000420808792, 0.03557068482041359, -0.025523461401462555, 0.023586146533489227, 0.03811969608068466, 0.05309145152568817, -0.010529199615120888, 0.07928898185491562, -0.03315209224820137, 0.02732252888381481, -0.08175461739301682, -0.039861589670181274, -0.06679204106330872, -0.014511581510305405, 0.22423890233039856, -0.005609406623989344, 0.06368155032396317, -0.020161857828497887, 0.022293759509921074, 0.07268688827753067, 0.04768749326467514, 0.018984152004122734, -0.014719628728926182, -0.031624000519514084, -0.03131784498691559, -0.0017137853428721428, -0.042614761739969254, -0.0012451704824343324, 0.06771665811538696, -0.01149524375796318, 0.01788502000272274, 0.04702160879969597, 0.0357765331864357, 0.14069227874279022, 0.019372258335351944, -0.011147385463118553, -0.20374667644500732, -0.004195220302790403, -0.011896897107362747, -0.04308205842971802, 0.031213538721203804, 0.06520138680934906, -0.021856235340237617, -0.008883864618837833, 0.06533157825469971, -0.07754222303628922, -0.028796013444662094, -0.05036091431975365, -0.054751958698034286, 0.0038944461848586798, -0.012615918181836605, 0.06735038012266159, -0.016643693670630455, 0.04564574733376503, 0.08920537680387497, 0.001100062858313322, 0.026013046503067017, -0.009029979817569256, -0.016705801710486412, 0.028004303574562073, -0.04430592805147171, 0.014125248417258263, -0.03422391414642334, 0.0788756012916565, -0.03831801190972328, -0.037770625203847885, -0.016434986144304276, 0.0009976865258067846, -0.0323617048561573, -0.06522216647863388, 0.008413477800786495, -0.0935741662979126, 0.03909887745976448, 0.04232612997293472, 0.03659188747406006, 0.02203298918902874, -0.00565719697624445, 0.010387931019067764, 0.004453467205166817, -0.0088711092248559, -0.04824152588844299, 0.00222714152187109, 0.039838969707489014, 0.004696051124483347, 0.009141637943685055, 0.003297225572168827, -0.009455963037908077, -0.09014299511909485, 0.01049747969955206, 0.021697763353586197, 0.046166058629751205, -0.07551838457584381, -0.0480598546564579, 0.06390725821256638, -0.32260480523109436, 0.019912201911211014, -0.07213730365037918, -0.06516619771718979, -0.012279761955142021, 0.07050950080156326, -0.2007194608449936, 0.04275752604007721, -0.021994221955537796, 0.028362944722175598, -0.026872744783759117, 0.2963635325431824, -0.04557427763938904, 0.0045049721375107765, -0.04736281931400299, -0.009104202501475811, -0.03255843743681908, -0.005085611715912819, 0.011967040598392487, 0.04976192116737366, 0.0191938616335392, -0.08612369000911713, -0.013043764978647232, -0.018677985295653343, 0.06078823655843735, -0.022849904373288155, -0.034091245383024216, 0.003043592907488346, -0.005943583324551582, 0.038452811539173126, 0.04437887296080589, -0.22873030602931976, -0.011979188770055771, 0.044343188405036926, 0.05481089651584625, 0.009740256704390049, -0.06255896389484406, -0.06582584977149963, 0.02503049187362194, 0.0014950663316994905, 0.007663790136575699, -0.00718929385766387, 0.0360758900642395, 0.018562883138656616, -0.4583660960197449, -0.027079233899712563, -0.18467184901237488, 0.0035128225572407246, -0.05865224450826645, -0.008259513415396214, 0.022909168154001236, 0.0375642254948616, -0.021402699872851372, -0.015300069004297256, -0.02300069108605385, -0.04215313494205475, -0.030762098729610443, 0.0665823370218277, 0.07500986009836197, -0.04695436358451843, 0.029156716540455818, -0.025217168033123016, -0.029532330110669136, 0.04126087948679924, 0.047646038234233856, 0.04516853392124176, -0.04800725355744362, -0.026557736098766327, 0.05792924389243126, -0.04108722135424614, 0.023203004151582718, 0.0074921054765582085, 0.0964353159070015, 0.024626553058624268, 0.019459392875432968, -0.019868621602654457, -0.029717931523919106, 0.014821067452430725, -0.09689079970121384, -0.09208132326602936, 0.0005997484549880028, -0.0002196409332100302, -0.03242310881614685, -0.03093790076673031, -0.02418428286910057, 0.19929762184619904, -0.08488059043884277, -0.00190876261331141, -0.013526247814297676, -0.015585302375257015, 0.04910113289952278, -0.026452895253896713, 0.006202384829521179, 0.016837337985634804, 0.03559812158346176, -0.03281727433204651, -0.04207354038953781, -0.007225838489830494, 0.10221371799707413, 0.005940940231084824, -0.010641681961715221, -0.0009305786225013435, -0.02834765426814556, 0.047375064343214035, -0.013287875801324844, 0.04352761432528496, -0.0006228100974112749, -0.008435416035354137, 0.0047037843614816666, 0.02024132013320923, -0.05719516798853874, -0.04551231861114502, 0.019194230437278748, 0.014000413939356804, 0.014926020987331867, 0.04625639691948891, -0.06619545817375183, 0.10913097113370895, -0.04744400084018707, 0.0344357006251812, -0.055866096168756485, -0.05221251770853996, 0.1956062912940979, -0.11517147719860077, 0.04355340451002121, -0.009099348448216915, -0.003882444929331541, 0.057190604507923126, -0.047253090888261795, -0.069283626973629, 0.0041439104825258255, 0.0020248894579708576, 0.037737004458904266, 0.03120964765548706, -0.1035144105553627, -0.07568478584289551, -0.08432945609092712, 0.003353149862959981, -0.026023879647254944, -0.03922176733613014, 0.02204141952097416, 0.006827107630670071, 0.003703031688928604, 0.021995792165398598, -0.005661001894623041, -0.05183941870927811, 0.012582586146891117, -0.11106478422880173, -0.2321361005306244, 0.007580309174954891, -0.021687699481844902, -0.04770306870341301, -0.029657987877726555, 0.008284148760139942, 0.10204525291919708, 0.0719456821680069, 0.01340375654399395, 0.24945802986621857, -0.029030796140432358, 0.03230923414230347, -0.035193007439374924, -0.0014327033422887325, -0.04603331908583641, 0.10080684721469879, -0.0006411723443306983, 0.03442942351102829, 0.0035577593371272087, -0.002149025909602642, 0.06196087971329689, -0.03758920356631279, -0.05997733399271965, -0.012662682682275772, -0.044622913002967834, -0.041623204946517944, 0.020130030810832977, -0.013730132952332497, -0.01707170344889164, 0.07818196713924408, 0.030053870752453804, -0.07028529047966003, -0.30279242992401123, -0.037469662725925446, -0.04795408993959427, -0.012372622266411781, 0.02478910982608795, -0.047366831451654434, 0.041070710867643356, -0.01865357719361782, -0.0331924669444561, 0.0071377139538526535, -0.040718235075473785, -0.030362509191036224, -0.03237789124250412, -0.023766355589032173, -0.03727859631180763, 0.025759372860193253, 0.017700664699077606, 0.012371034361422062, -0.03712233528494835, 0.04742475971579552, -0.020432883873581886, -0.04366379231214523, 0.008605479262769222, 0.08102196455001831, -0.20951807498931885, 0.10575616359710693, -0.01947949454188347, 0.015602098777890205, 0.00217430223710835, -0.026434125378727913, -0.03493449091911316, -0.1232689693570137, 0.017161667346954346, -0.013046568259596825, -0.03224029764533043, 0.06346827000379562, 0.002288462594151497, 0.004833023529499769, 0.010994665324687958, -0.026157094165682793, -0.02593374252319336, 0.029332557693123817, 0.018249358981847763, -0.007014033850282431, -0.011393860913813114, 0.011908302083611488, 0.039434876292943954, -0.0400676392018795, -0.07091765105724335, 0.49785497784614563, -0.026920268312096596, 0.009746892377734184, 0.019901612773537636, 0.18367773294448853, 0.005276066716760397, 0.014876918867230415, -0.02912994660437107, -0.06688132882118225, -0.015318105928599834, -0.01483045145869255, 0.1554793119430542, 0.06607776880264282, -0.04990667477250099, 0.03816739469766617, -0.01069364883005619, -0.10834738612174988, -0.02378138341009617, -0.05725714564323425, -0.025217929854989052, -0.05267300456762314, 0.053809914737939835, -0.04634804278612137, 0.09239423274993896, 0.00947287492454052, 0.004061272833496332, 0.0169654730707407, -0.053057026118040085, -0.016948726028203964, 0.051149532198905945, -0.02401314489543438, -0.003292106557637453, -0.10382527112960815, -0.04795597866177559, -0.021892782300710678, 0.024799969047307968, 0.03483009710907936, -0.015789616852998734, -0.0036111650988459587, -0.12697412073612213, -0.04390875622630119, 0.03709934651851654, -0.040146466344594955, -0.009510749019682407, 0.07805352658033371, -0.004416113719344139, 0.021227357909083366, -0.020142458379268646, -0.07218810170888901, -0.014361330308020115, 0.0206767525523901, -0.030596360564231873, -0.018818449229002, -0.020437220111489296, -0.0269027017056942, -0.04633314162492752, -0.015268487855792046, -0.11509953439235687], 'sd': [0.0945843905210495, 0.08281105756759644, 0.08855526894330978, 0.09188363701105118, 0.08729825913906097, 0.09090027958154678, 0.06252261996269226, 0.08591415733098984, 0.07301259785890579, 0.09046589583158493, 0.09427208453416824, 0.11945832520723343, 0.1352987140417099, 0.10044534504413605, 0.08082167059183121, 0.07916606962680817, 0.09169785678386688, 0.08845368027687073, 0.10096300393342972, 0.07477933168411255, 0.07782401889562607, 0.08127758651971817, 0.08797422796487808, 0.08409426361322403, 0.08648012578487396, 0.09822618961334229, 0.07172342389822006, 0.0992337167263031, 0.09153959900140762, 0.08348765969276428, 0.08432792872190475, 0.10107677429914474, 0.09116517752408981, 0.0874406024813652, 0.0906476229429245, 0.08486215770244598, 0.1634405553340912, 0.11185014992952347, 0.08075163513422012, 0.06153954565525055, 0.08816948533058167, 0.12223705649375916, 0.11931715905666351, 0.07735168188810349, 0.13397221267223358, 0.0808684229850769, 0.1573573499917984, 0.09770303219556808, 0.07751967012882233, 0.11298120021820068, 0.09737983345985413, 0.0838470533490181, 0.13894028961658478, 0.08728764206171036, 0.05447669327259064, 0.07640805095434189, 0.12466385215520859, 0.07484377175569534, 0.08585388213396072, 0.0786919891834259, 0.0811500996351242, 0.07970266044139862, 0.08278129994869232, 0.09672024101018906, 0.07681208103895187, 0.06825139373540878, 0.07505404204130173, 0.08219411969184875, 0.11345981806516647, 0.07074417173862457, 0.07839976251125336, 0.12566348910331726, 0.07582125067710876, 0.07656680792570114, 0.08201708644628525, 0.07742483168840408, 0.10558978468179703, 0.09071901440620422, 0.08173556625843048, 0.08077003806829453, 0.0786154642701149, 0.08369942754507065, 0.08618403971195221, 0.098020039498806, 0.08670134097337723, 0.07959605753421783, 0.07969468832015991, 0.083461694419384, 0.06764213740825653, 0.08462885022163391, 0.08090706914663315, 0.07513213157653809, 0.0818512886762619, 0.09055961668491364, 0.08684065192937851, 0.07553412765264511, 0.07441035658121109, 0.08767950534820557, 0.09249072521924973, 0.07551687955856323, 0.10402436554431915, 0.08404820412397385, 0.12493398785591125, 0.09644767642021179, 0.09739916771650314, 0.09665591269731522, 0.07690344005823135, 0.08651761710643768, 0.08111605793237686, 0.10427581518888474, 0.10238838940858841, 0.07435742020606995, 0.08354002982378006, 0.0791458711028099, 0.08431261032819748, 0.07556962966918945, 0.08944554626941681, 0.08670081943273544, 0.14967481791973114, 0.07532314211130142, 0.08910038322210312, 0.07211492210626602, 0.10919473320245743, 0.08338772505521774, 0.1012311652302742, 0.08702777326107025, 0.07394441962242126, 0.07311154156923294, 0.08571575582027435, 0.09773619472980499, 0.09413252770900726, 0.08261186629533768, 0.07376605272293091, 0.07753811031579971, 0.08574097603559494, 0.13721711933612823, 0.08190017938613892, 0.09050516784191132, 0.11609666049480438, 0.0758044570684433, 0.08356354385614395, 0.09868057072162628, 0.11806779354810715, 0.10985857993364334, 0.09556933492422104, 0.11136705428361893, 0.08841776847839355, 0.08185748010873795, 0.07446204870939255, 0.08361148089170456, 0.08578448742628098, 0.08385023474693298, 0.09645510464906693, 0.09440067410469055, 0.07339394837617874, 0.080130934715271, 0.08416348695755005, 0.0746631771326065, 0.09568706154823303, 0.07761533558368683, 0.07480193674564362, 0.07945719361305237, 0.1141248568892479, 0.09578882902860641, 0.07597804814577103, 0.08744429051876068, 0.09249939024448395, 0.07951664179563522, 0.08223016560077667, 0.12542115151882172, 0.10912385582923889, 0.09248772263526917, 0.07485706359148026, 0.12958881258964539, 0.08105061203241348, 0.0947147011756897, 0.09245684742927551, 0.07397531718015671, 0.09867431968450546, 0.06616425514221191, 0.082825668156147, 0.07996992021799088, 0.0931205153465271, 0.07720588147640228, 0.0978076383471489, 0.08227938413619995, 0.08546208590269089, 0.1328490823507309, 0.09079831093549728, 0.08782509714365005, 0.08479009568691254, 0.08562621474266052, 0.09249643981456757, 0.08575040102005005, 0.07681046426296234, 0.07807909697294235, 0.08534727245569229, 0.10321514308452606, 0.09705597162246704, 0.08861280232667923, 0.08337949216365814, 0.09157142043113708, 0.08908937871456146, 0.07627470791339874, 0.07985378801822662, 0.0922449603676796, 0.0773622915148735, 0.08784470707178116, 0.07168852537870407, 0.11163941025733948, 0.089979387819767, 0.09181611239910126, 0.08984997123479843, 0.09331639856100082, 0.08141275495290756, 0.08228405565023422, 0.08122138679027557, 0.07090194523334503, 0.1309875100851059, 0.07262436300516129, 0.14236034452915192, 0.08632688224315643, 0.0843597799539566, 0.08039091527462006, 0.08482944965362549, 0.07391979545354843, 0.11161229014396667, 0.07011360675096512, 0.08882416039705276, 0.0951770544052124, 0.09992729872465134, 0.10742282122373581, 0.1444195955991745, 0.07917791604995728, 0.08759153634309769, 0.12177465111017227, 0.12381356209516525, 0.10424258559942245, 0.11183558404445648, 0.10597160458564758, 0.10661473125219345, 0.11279111355543137, 0.07783173769712448, 0.08421660214662552, 0.07912326604127884, 0.08475925028324127, 0.0855550542473793, 0.08639801293611526, 0.10252057760953903, 0.13520042598247528, 0.12300128489732742, 0.08011182397603989, 0.094731904566288, 0.07253019511699677, 0.08540316671133041, 0.08507820963859558, 0.07051965594291687, 0.0858454778790474, 0.07628891617059708, 0.08467424660921097, 0.09849121421575546, 0.12603439390659332, 0.08721188455820084, 0.08370902389287949, 0.09523782879114151, 0.08347311615943909, 0.08254066854715347, 0.07380582392215729, 0.09027891606092453, 0.07962393015623093, 0.13877169787883759, 0.09667619317770004, 0.12035609781742096, 0.04408353939652443, 0.10575789213180542, 0.07774375379085541, 0.0829700231552124, 0.13235922157764435, 0.0781586766242981, 0.08508597314357758, 0.08202312886714935, 0.0880134180188179, 0.08372996002435684, 0.11826027184724808, 0.10078071802854538, 0.11588189750909805, 0.09572300314903259, 0.07246505469083786, 0.10240800678730011, 0.14318856596946716, 0.0831088200211525, 0.0836879312992096, 0.07905691862106323, 0.0886685773730278, 0.09115175157785416, 0.08867252618074417, 0.12327240407466888, 0.07684611529111862, 0.09440254420042038, 0.08763287216424942, 0.09989725053310394, 0.08236172795295715, 0.09246090799570084, 0.09185408055782318, 0.08284742385149002, 0.07842779904603958, 0.08969611674547195, 0.08195719867944717, 0.07418984174728394, 0.1096833124756813, 0.07810898125171661, 0.09300515055656433, 0.08264853805303574, 0.12859205901622772, 0.09332259744405746, 0.07441636174917221, 0.1288992166519165, 0.10025552660226822, 0.09224723279476166, 0.06782549619674683, 0.08470480144023895, 0.12760180234909058, 0.09662596136331558, 0.08136573433876038, 0.09120947122573853, 0.0781783014535904, 0.12551359832286835, 0.08850686997175217, 0.08939201384782791, 0.09660399705171585, 0.08141274005174637, 0.1427324116230011, 0.08720079809427261, 0.08681102097034454, 0.08371549099683762, 0.0914575383067131, 0.09490204602479935, 0.08102111518383026, 0.07767379283905029, 0.09075881540775299, 0.08360784500837326, 0.0820063054561615, 0.0911417230963707, 0.07194674760103226, 0.08228117227554321, 0.07672103494405746, 0.1258106678724289, 0.07761790603399277, 0.12115978449583054, 0.10514474660158157, 0.08297393471002579, 0.08754289895296097, 0.0917004942893982, 0.08478046208620071, 0.09403293579816818, 0.08415374904870987, 0.08331133425235748, 0.09344794601202011, 0.08649927377700806, 0.11275175213813782, 0.07307630777359009, 0.10935694724321365, 0.1447729766368866, 0.09466011077165604, 0.08235099166631699, 0.0952937975525856, 0.11411856859922409, 0.08123917132616043, 0.0979243814945221, 0.07548047602176666, 0.10042135417461395, 0.10223664343357086, 0.06005650758743286, 0.08484742045402527, 0.14140313863754272, 0.07857639342546463, 0.11249393969774246, 0.09092489629983902, 0.09187249839305878, 0.19597235321998596, 0.0753389224410057, 0.0753864049911499, 0.11859596520662308, 0.0768309086561203, 0.08122828602790833, 0.1399894654750824, 0.07581882923841476, 0.15436357259750366, 0.09691183269023895, 0.0762137770652771, 0.09486947953701019, 0.04209841787815094, 0.07903751730918884, 0.10508930683135986, 0.09552863985300064, 0.0851389542222023, 0.08536989986896515, 0.07962439209222794, 0.0813678577542305, 0.07548189163208008, 0.07450694590806961, 0.07490622997283936, 0.07851092517375946, 0.07820039242506027, 0.08449588716030121, 0.08492586761713028, 0.09070289134979248, 0.08172624558210373, 0.07255098968744278, 0.0836959257721901, 0.07902554422616959, 0.08311489224433899, 0.08286531269550323, 0.0789472684264183, 0.08935187011957169, 0.08707243949174881, 0.09463963657617569, 0.08529467135667801, 0.05144328251481056, 0.11051595956087112, 0.10820488631725311, 0.08734717220067978, 0.08203711360692978, 0.14234866201877594, 0.07643914967775345, 0.08452009409666061, 0.0791277140378952, 0.08017551153898239, 0.08006601780653, 0.1006118655204773, 0.08065317571163177, 0.07886698842048645, 0.1013680100440979, 0.08309702575206757, 0.12220461666584015, 0.09100817888975143, 0.07756925374269485, 0.07798050343990326, 0.15712229907512665, 0.09545929729938507, 0.0997982919216156, 0.12542656064033508, 0.07905974239110947, 0.12338801473379135, 0.07914754003286362, 0.09470940381288528, 0.08802197128534317, 0.08429297804832458, 0.0785159170627594, 0.07396913319826126, 0.09984025359153748, 0.07940080016851425, 0.0819186270236969, 0.09092622250318527, 0.08764839917421341, 0.08357313275337219, 0.10248948633670807, 0.10790859907865524, 0.13262349367141724, 0.08162951469421387, 0.08417025953531265, 0.07854123413562775, 0.07831409573554993, 0.08152992278337479, 0.07888349890708923, 0.09111932665109634, 0.09762189537286758, 0.0843866840004921, 0.0937931016087532, 0.0832853689789772, 0.10184085369110107, 0.08878692984580994, 0.09115982800722122, 0.08995668590068817, 0.08291700482368469, 0.08682626485824585, 0.08268208801746368, 0.0794336125254631, 0.06700584292411804, 0.08797986060380936, 0.09709509462118149, 0.10763303935527802, 0.08554510027170181, 0.08936340361833572, 0.09945391118526459, 0.07516220957040787, 0.07757201045751572, 0.10400499403476715, 0.10122940689325333, 0.08236062526702881, 0.0771460086107254, 0.09529951959848404, 0.1423654705286026, 0.08495345711708069, 0.08742937445640564, 0.09360009431838989, 0.07887637615203857, 0.09005507826805115, 0.08190149068832397, 0.07386543601751328, 0.09090284258127213, 0.0990501195192337, 0.1162995845079422, 0.05393919721245766, 0.07530006766319275, 0.10581761598587036, 0.07357810437679291, 0.09346930682659149, 0.09113588184118271, 0.09375623613595963, 0.08378180116415024, 0.0752219706773758, 0.07066473364830017, 0.08091619610786438, 0.09443473815917969, 0.09337407350540161, 0.09797948598861694, 0.0860624685883522, 0.11600706726312637, 0.08924780786037445, 0.07425444573163986, 0.0808379128575325, 0.08264902979135513, 0.10777219384908676, 0.07988713681697845, 0.13370198011398315, 0.09865391999483109, 0.13725696504116058, 0.08029326796531677, 0.08826630562543869, 0.12393151968717575, 0.09417426586151123, 0.0743182823061943, 0.09096019715070724, 0.07856026291847229, 0.09974709153175354, 0.08074752241373062, 0.08204706758260727, 0.09892232716083527, 0.08837844431400299, 0.08321469277143478, 0.08062109351158142, 0.0868590846657753, 0.0844016820192337, 0.07973130792379379, 0.07858423888683319, 0.0819278433918953, 0.07799989730119705, 0.08980049937963486, 0.07561886310577393, 0.1095031350851059, 0.09280482679605484, 0.08388498425483704, 0.11926300078630447, 0.07920236140489578, 0.08258175849914551, 0.07903313636779785, 0.08121302723884583, 0.09190717339515686, 0.11748582124710083, 0.1023666188120842, 0.08388619869947433, 0.09753262996673584, 0.08845063298940659, 0.09092975407838821, 0.09818151593208313, 0.056513462215662, 0.09070383012294769, 0.08084933459758759, 0.07889536023139954, 0.08496995270252228, 0.09163472056388855, 0.08115093410015106, 0.08039627969264984, 0.08829908072948456, 0.08026619255542755, 0.08011764287948608, 0.09087889641523361, 0.07799883931875229, 0.11228520423173904, 0.08171015977859497, 0.0776924416422844, 0.09838175773620605, 0.08325918763875961, 0.07905381172895432, 0.07832539081573486, 0.08443272858858109, 0.07443516701459885, 0.07766902446746826, 0.08388571441173553, 0.08819614350795746, 0.10076096653938293, 0.08659990131855011, 0.07663881033658981, 0.07311726361513138, 0.07577891647815704, 0.09311471879482269, 0.07726029306650162, 0.12165877223014832, 0.09320054203271866, 0.10624304413795471, 0.09032130986452103, 0.0921282023191452, 0.1137898713350296, 0.08187471330165863, 0.07783099263906479, 0.08069642633199692, 0.08213239908218384, 0.10769562423229218, 0.08332797884941101, 0.08443067222833633, 0.09433802962303162, 0.08334832638502121, 0.10397770255804062, 0.0792081356048584, 0.1085929200053215, 0.08099294453859329, 0.12502679228782654, 0.09048503637313843, 0.08464992046356201, 0.0817720890045166, 0.07744437456130981, 0.06991100311279297, 0.10924883186817169, 0.07679609209299088, 0.08292728662490845, 0.08799004554748535, 0.10640816390514374, 0.08240584284067154, 0.1297818422317505, 0.10618170350790024, 0.07723493874073029, 0.08662614226341248, 0.1003507450222969, 0.07379171252250671, 0.11809422820806503, 0.08039337396621704, 0.08546019345521927, 0.08836717903614044, 0.08260404318571091, 0.08670143038034439, 0.08919281512498856, 0.07487766444683075, 0.08813522756099701, 0.10829176008701324, 0.09793984889984131, 0.08537880331277847, 0.07946841418743134, 0.07864994555711746, 0.08892574906349182, 0.07897654175758362, 0.07278870046138763, 0.07299281656742096, 0.08331789076328278, 0.11904490739107132, 0.10789169371128082, 0.08279043436050415, 0.08889850974082947, 0.10620716214179993, 0.08905428647994995, 0.11539307981729507, 0.07652653008699417, 0.10043857246637344, 0.07642537355422974, 0.10086409747600555, 0.09296207129955292, 0.07727839797735214, 0.08212604373693466, 0.08510191738605499, 0.08345995843410492, 0.08385264128446579, 0.09598927944898605, 0.07422246783971786, 0.07797667384147644, 0.10304664820432663, 0.0794416293501854, 0.07637052237987518, 0.08697133511304855, 0.09490664303302765, 0.07970096915960312, 0.07642998546361923, 0.09382285177707672, 0.09802838414907455, 0.0809132382273674, 0.08405052125453949, 0.20543904602527618, 0.08480542153120041, 0.09770739823579788, 0.07386523485183716, 0.07462660223245621, 0.08761154860258102, 0.08871619403362274, 0.1433931291103363, 0.09955295920372009, 0.0755114033818245, 0.08220866322517395, 0.11026058346033096, 0.08071742951869965, 0.07945073395967484, 0.08192797005176544, 0.07545856386423111, 0.08273623883724213, 0.10868469625711441, 0.07843025773763657, 0.08794773370027542, 0.0840781033039093, 0.08628524839878082, 0.10419433563947678, 0.10298458486795425, 0.07603919506072998, 0.12062105536460876, 0.09178751707077026, 0.09067121148109436, 0.08598102629184723, 0.10967127978801727, 0.0749160498380661, 0.10745742172002792, 0.08925539255142212, 0.08077160269021988, 0.09085965901613235, 0.0745173990726471, 0.1388251781463623, 0.07732266187667847, 0.08651905506849289, 0.1077767014503479, 0.09790831059217453, 0.09195929765701294, 0.09466920793056488, 0.08529075235128403, 0.09165368974208832, 0.08320768922567368, 0.07959745079278946, 0.0812944546341896, 0.12423603981733322, 0.08152782917022705, 0.07428593188524246, 0.09354741871356964, 0.08886891603469849, 0.08126457780599594, 0.10286517441272736, 0.0799025297164917, 0.09567543864250183, 0.08645270019769669, 0.0831863209605217, 0.09113594144582748, 0.08324306458234787, 0.07536345720291138, 0.08198904246091843, 0.08295488357543945, 0.0835072249174118, 0.08735757321119308, 0.07581783086061478, 0.10555560141801834, 0.0830002874135971, 0.0783148780465126, 0.09049134701490402, 0.10028024017810822, 0.08611784130334854, 0.09304846823215485, 0.07671061903238297, 0.08202549815177917, 0.11719613522291183, 0.09008724987506866, 0.08674905449151993, 0.08725167065858841, 0.10691925883293152, 0.07512491941452026, 0.08926992863416672]}
FRESH = CONST["fresh"]
ORDER1 = ['task1572_samsum_summary', 'task363_sst2_polarity_classification', 'task1290_xsum_summarization', 'task181_outcome_extraction', 'task002_quoref_answer_generation', 'task1510_evalution_relation_extraction', 'task639_multi_woz_user_utterance_generation', 'task1729_personachat_generate_next', 'task073_commonsenseqa_answer_generation', 'task1590_diplomacy_text_generation', 'task748_glucose_reverse_cause_event_detection', 'task511_reddit_tifu_long_text_summarization', 'task591_sciq_answer_generation', 'task1687_sentiment140_classification', 'task875_emotion_classification']
ORDER2 = ['task748_glucose_reverse_cause_event_detection', 'task073_commonsenseqa_answer_generation', 'task1590_diplomacy_text_generation', 'task639_multi_woz_user_utterance_generation', 'task1572_samsum_summary', 'task1687_sentiment140_classification', 'task591_sciq_answer_generation', 'task363_sst2_polarity_classification', 'task1510_evalution_relation_extraction', 'task1729_personachat_generate_next', 'task181_outcome_extraction', 'task511_reddit_tifu_long_text_summarization', 'task002_quoref_answer_generation', 'task1290_xsum_summarization', 'task875_emotion_classification']
LONG3 = "yelp,amazon,mnli,cb,copa,qqp,rte,imdb,sst2,dbpedia,agnews,yahoo,multirc,boolq,wic".split(",")
log("pool sha1", CONST["pool_sha1"], "| fresh tasks", len(FRESH))

## The rules and the measures (copied verbatim from `paper_certified/hidden_rules_study.py`)

In [ ]:
def mse(v): v = np.array(v, float); return float(v.mean()), float(v.std(ddof=1) / math.sqrt(len(v)))


def standardiser(P):
    mu = P.mean(0); sd = P.std(0); sd[sd < 1e-6] = 1.0
    return lambda X: ((X - mu) / sd).astype(np.float64)


def routes_joint(fit, cal, te, C):
    """R_te[j][:, T], R_cal[j][:, T] = route after tasks 0..T have arrived (-1 before the task's own arrival)."""
    K = len(fit); R_te = [np.full((len(te[j]), K), -1) for j in range(K)]; R_cal = [np.full((len(cal[j]), K), -1) for j in range(K)]
    for T in range(K):
        if T == 0:
            pred = lambda X: np.zeros(len(X), int)
        else:
            X = np.vstack(fit[:T + 1]); y = np.concatenate([np.full(len(fit[k]), k) for k in range(T + 1)])
            clf = LogisticRegression(C=C, class_weight="balanced", max_iter=3000).fit(X, y)
            pred = clf.predict
        for j in range(T + 1):
            R_te[j][:, T] = pred(te[j]); R_cal[j][:, T] = pred(cal[j])
    return R_te, R_cal


def ridge_sums(fit):
    """Per task: weighted sums of x x^T and of x (a constant 1 is appended), each task with the same total weight 200."""
    out = []
    for X in fit:
        Z = np.hstack([X, np.ones((len(X), 1))]); w = 200.0 / len(Z)
        out.append((w * Z.T @ Z, w * Z.sum(0)))
    return out


def routes_ridge(fit, cal, te, lam, order=None):
    K = len(fit); sums = ridge_sums(fit); d = sums[0][0].shape[0]
    R_te = [np.full((len(te[j]), K), -1) for j in range(K)]; R_cal = [np.full((len(cal[j]), K), -1) for j in range(K)]
    G = np.zeros((d, d)); Cm = np.zeros((d, K)); W_last = None
    add = lambda X: np.hstack([X, np.ones((len(X), 1))])
    for T in range(K):
        G += sums[T][0]; Cm[:, T] = sums[T][1]
        W = np.linalg.solve(G + lam * (d - 1) * np.eye(d), Cm[:, :T + 1]); W_last = W
        for j in range(T + 1):
            R_te[j][:, T] = (add(te[j]) @ W).argmax(1); R_cal[j][:, T] = (add(cal[j]) @ W).argmax(1)
    return R_te, R_cal, W_last


def routes_handicap(fit, cal, te, lam, t_list, floor=False):
    """Rule H (PLAN.md 07:34): the joint ridge recogniser at every arrival; the newcomer takes x only if its margin over the best older task
    exceeds theta = the largest, over older tasks, of the (t+1)-th largest margin among that task's held-out inputs. Newest taker wins."""
    K = len(fit); sums = ridge_sums(fit); d = sums[0][0].shape[0]; add = lambda X: np.hstack([X, np.ones((len(X), 1))])
    G = np.zeros((d, d)); Cm = np.zeros((d, K)); M_te = [np.full((len(te[j]), K), np.inf) for j in range(K)]; M_cal = [np.full((len(cal[j]), K), np.inf) for j in range(K)]
    for l in range(K):
        G += sums[l][0]; Cm[:, l] = sums[l][1]
        if l == 0: continue
        W = np.linalg.solve(G + lam * (d - 1) * np.eye(d), Cm[:, :l + 1])
        mg = lambda X: (lambda Sx: Sx[:, l] - Sx[:, :l].max(1))(add(X) @ W)
        for j in range(K):
            M_te[j][:, l] = mg(te[j]); M_cal[j][:, l] = mg(cal[j])
    out = {}
    for t in t_list:
        theta = np.full(K, -np.inf)
        for l in range(1, K):
            theta[l] = max((np.sort(M_cal[j][:, l])[::-1][t] if t < len(M_cal[j]) else -np.inf) for j in range(l))
            if floor: theta[l] = max(theta[l], 0.0)      # rule H': the newcomer never takes an input that the joint recogniser gives to an older task
        def route(Mx, T):
            r = np.zeros(len(Mx), int)
            for l in range(1, T + 1): r = np.where(Mx[:, l] > theta[l], l, r)
            return r
        R_te = [np.full((len(te[j]), K), -1) for j in range(K)]; R_cal = [np.full((len(cal[j]), K), -1) for j in range(K)]
        for j in range(K):
            for T in range(j, K):
                R_te[j][:, T] = route(M_te[j], T); R_cal[j][:, T] = route(M_cal[j], T)
        take = [1.0] + [float((M_cal[l][:, l] > theta[l]).mean()) for l in range(1, K)]
        out[t] = (R_te, R_cal, {"takes_of_own_heldout": take, "theta": [None] + [float(x) for x in theta[1:]]})
    return out


def measures(names, R_te, R_cal, extra=None, held_out_used_by_rule=False, r=None, path=False, handicap_t=None):
    K = len(names); per = {}
    for j, kj in enumerate(names):
        te, ca = R_te[j], R_cal[j]; m = len(ca); n = len(te)
        own_arr = float((te[:, j] == j).mean()); own_end = float((te[:, K - 1] == j).mean())
        ch_end = te[:, j] != te[:, K - 1]; ch_ever = (te[:, j:] != te[:, [j]]).any(1)
        c_end = int((ca[:, j] != ca[:, K - 1]).sum()); c_ever = int((ca[:, j:] != ca[:, [j]]).any(1).sum())
        wrong = te[:, K - 1][te[:, K - 1] != j]
        d = {"n_test": n, "m": m, "own_at_arrival": own_arr, "own_at_end": own_end, "changed": float(ch_end.mean()), "changed_ever": float(ch_ever.mean()),
             "c": c_end, "c_ever": c_ever, "certificate": (c_end + 1) / (m + 1), "certificate_ever": (c_ever + 1) / (m + 1),
             "upper95": float(beta_dist.ppf(0.95, c_end + 1, m - c_end)) if c_end < m else 1.0,
             "heldout_own_at_arrival": float((ca[:, j] == j).mean()), "heldout_own_at_end": float((ca[:, K - 1] == j).mean()),
             "prior_bound": (1 - own_arr) + (1 - own_end)}
        if len(wrong):
            cnt = np.bincount(wrong, minlength=K); d["top_wrong_destination"] = names[int(cnt.argmax())]; d["top_wrong_destination_share_of_test"] = float(cnt.max() / n)
        if r is not None:
            d["claims_bound"] = (1 - extra["coverage_at_arrival"][j]) + r * (K - 1 - j) / (m + 1); d["coverage_at_arrival"] = extra["coverage_at_arrival"][j]
        if handicap_t is not None:
            d["handicap_bound"] = (handicap_t + 1) * (K - 1 - j) / (m + 1); d["takes_of_own_heldout"] = extra["takes_of_own_heldout"][j]
        if path:
            d["path_changed_test"] = [float((te[:, j] != te[:, T]).mean()) for T in range(j, K)]
            d["path_certificate"] = [float(((ca[:, j] != ca[:, T]).sum() + 1) / (m + 1)) for T in range(j, K)]
        per[kj] = d
    keys = ["own_at_arrival", "own_at_end", "changed", "changed_ever", "certificate", "certificate_ever", "upper95", "heldout_own_at_arrival", "heldout_own_at_end", "prior_bound"]
    if r is not None: keys += ["claims_bound", "coverage_at_arrival"]
    if handicap_t is not None: keys += ["handicap_bound", "takes_of_own_heldout"]
    mean, se = {}, {}
    for k in keys: mean[k], se[k] = mse([per[t][k] for t in names])
    return {"per_task": per, "mean": mean, "se": se, "certificate_valid": not held_out_used_by_rule}


def fresh_split():
    perm = list(range(800)); random.Random(0).shuffle(perm); sel = perm[:400]
    return sel[:200], sel[200:], list(range(800, 1000))

def fingerprint(inputs):
    return hashlib.sha1("\x1e".join(inputs).encode("utf-8", errors="surrogatepass")).hexdigest()

def standardiser_embedded(mu, sd):
    """The same expression as standardiser(), with the pool mean and standard deviation (float32) given."""
    mu = np.array(mu, dtype=np.float32); sd = np.array(sd, dtype=np.float32)
    return lambda X: ((X - mu) / sd).astype(np.float64)

## Data: the official repository (SuperNI and Long Sequence) and the 90 fresh tasks

In [ ]:
def sh(cmd, cwd=None, tries=1, wait=20):
    for k in range(tries):
        r = subprocess.run(cmd, cwd=cwd, shell=True, capture_output=True, text=True)
        if r.returncode == 0:
            return r.stdout
        log(f"command failed ({k + 1}/{tries}): {cmd[:90]} -> {r.stderr.strip()[-300:]}")
        if k + 1 < tries: time.sleep(wait * (k + 1))
    raise RuntimeError(f"command failed after {tries} tries: {cmd}\n{r.stderr[-500:]}")

def get_repo():
    d = os.path.join(WORK, "gainlora")
    if os.path.isdir(os.path.join(d, "CL_Benchmark")):
        try:
            if sh("git rev-parse HEAD", cwd=d).strip() == OFFICIAL_COMMIT: return d
        except Exception: pass
        shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d, exist_ok=True)
    try:
        sh("git init -q && git remote add origin " + OFFICIAL_URL, cwd=d)
        sh(f"git fetch -q --depth 1 origin {OFFICIAL_COMMIT} && git checkout -q FETCH_HEAD", cwd=d, tries=3)
    except Exception as e:
        log("shallow fetch of the commit failed, full clone instead:", str(e)[:200])
        shutil.rmtree(d, ignore_errors=True)
        sh(f"git clone -q {OFFICIAL_URL} {d}", tries=3)
        sh(f"git checkout -q {OFFICIAL_COMMIT}", cwd=d)
    assert sh("git rev-parse HEAD", cwd=d).strip() == OFFICIAL_COMMIT
    return d

REPO = os.environ.get("HS_LOCAL_REPO") or get_repo()      # HS_LOCAL_REPO: test hook only (a local copy of the same commit)
BENCH = os.path.join(REPO, "CL_Benchmark")
log("official repository at", REPO, "| commit", OFFICIAL_COMMIT)

# the order of Long Sequence order 4: the task_config_dir entries of the official script, in order
_txt = open(os.path.join(REPO, "gen_script_long_order4_t5_inflora.sh")).read()
LONG4 = re.findall(r"--task_config_dir\s+\S*?/([A-Za-z0-9_]+)(?=\s|\\|$)", _txt)
assert len(LONG4) == 15 and set(LONG4) == set(LONG3), LONG4
log("Long Sequence order 4:", ",".join(LONG4))

def load_inputs(folder, name, split):
    d = json.load(open(os.path.join(BENCH, folder, name, split + ".json"), encoding="utf-8"))
    return [str(i["input"]) for i in d["Instances"]]      # the Definition is never read

# ---- the 90 fresh tasks: same Range request and same parser as paper_certified/fresh_tasks_fetch.py
def objects(text, key):
    """The objects of the list that follows "key": [ ... as far as the fetched part of the file goes."""
    m = re.search(r'"%s"\s*:\s*\[' % re.escape(key), text)
    out, dec = [], json.JSONDecoder()
    if not m:
        return out
    j = m.end()
    while True:
        while j < len(text) and text[j] in " \n\r\t,":
            j += 1
        if j >= len(text) or text[j] != "{":
            return out
        try:
            obj, j = dec.raw_decode(text, j)
        except ValueError:
            return out
        out.append(obj)

HEAD, NINST = 12 * 1024 * 1024, 1000

def fetch_text(name, attempts=6):
    cmd = ["curl", "-sS", "-f", "-L", "--max-time", "300", "--retry", "2", "-H", f"Range: bytes=0-{HEAD - 1}", NI_RAW + f"tasks/{name}.json"]
    err = ""
    for k in range(attempts):
        r = subprocess.run(cmd, capture_output=True)
        if r.returncode == 0 and r.stdout:
            return r.stdout.decode("utf-8", errors="ignore")
        err = r.stderr.decode(errors="ignore").strip()[-200:] or "empty answer"
        wait = min(150, 10 * 2 ** k)            # GitHub rate limits (HTTP 429/403) and transient errors: wait longer each time
        log(f"download of {name}: attempt {k + 1}/{attempts} failed ({err}); waiting {wait}s")
        time.sleep(wait)
    raise RuntimeError(f"download of {name} failed after {attempts} attempts: {err}")

def parse_inputs(text):
    inst = []
    for o in objects(text, "Instances"):
        if isinstance(o.get("input"), str) and o["input"].strip():
            out0 = o.get("output")
            out0 = out0[0] if isinstance(out0, list) and out0 else out0
            if isinstance(out0, str):
                inst.append(o["input"])
    return inst

CPU_FRESH_DIR = os.environ.get("HS_CPU_FRESH")          # test hook only: use CPU copies for the tasks beyond the first HS_DL_LIMIT
DL_LIMIT = int(os.environ.get("HS_DL_LIMIT", "90"))
os.makedirs(os.path.join(WORK, "fresh"), exist_ok=True)
FRESH_INPUTS, DATA_DIFFERS, DOWNLOAD_FAILED, MATCH = {}, [], [], 0
for i, (name, n_cpu, sha_cpu) in enumerate(FRESH):
    cache = os.path.join(WORK, "fresh", f"{i:02d}_{name}.json")
    try:
        if i >= DL_LIMIT and CPU_FRESH_DIR:
            allt = {t["name"]: t for f in ("fresh_tasks.json", "fresh_tasks_2.json", "fresh_tasks_3.json") for t in json.load(open(os.path.join(CPU_FRESH_DIR, f)))["tasks"]}
            inputs = [str(x["input"]) for x in allt[name]["instances"]]
        elif os.path.exists(cache):
            inputs = json.load(open(cache))
        else:
            inputs = parse_inputs(fetch_text(name)); json.dump(inputs, open(cache + ".tmp", "w")); os.replace(cache + ".tmp", cache)
    except Exception as e:
        log("WARNING:", e); DOWNLOAD_FAILED.append(name); continue
    inputs = inputs[:NINST]; FRESH_INPUTS[name] = inputs
    same = len(inputs) == n_cpu and fingerprint(inputs) == sha_cpu
    MATCH += same
    if not same: DATA_DIFFERS.append({"name": name, "parsed": len(inputs), "expected": n_cpu})
    if i % 10 == 9 or i == len(FRESH) - 1: log(f"fresh tasks {i + 1}/{len(FRESH)} read; fingerprints matching so far {MATCH}")
log(f"FINGERPRINT CHECK: {MATCH} of {len(FRESH)} fresh tasks identical to the CPU files; data_differs = {len(DATA_DIFFERS)}; download_failed = {len(DOWNLOAD_FAILED)}")
if DATA_DIFFERS: print("data_differs:", DATA_DIFFERS)
if DOWNLOAD_FAILED: print("WARNING: not downloaded, left out of every sequence:", DOWNLOAD_FAILED)

## Splits and features
Benchmark tasks: n training inputs, `perm` = seed-0 shuffle, `sel` = first min(n, 400); held-out = the first m of `sel` (m = 200 if n >= 400 else n // 2), fit = the
rest; test = all official test inputs. Fresh tasks: `fresh_split()` (seed-0 shuffle of range(800), first 400: 200 held-out, then 200 fit; test = instances 800..999).

In [ ]:
import torch
from transformers import AutoTokenizer, T5EncoderModel
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "t5-small" if QUICK else "t5-base"
if DEVICE == "cpu" and not QUICK: log("WARNING: no GPU found; the encoding of about 150,000 inputs on a CPU will take hours")
tok = AutoTokenizer.from_pretrained(MODEL_NAME); model = T5EncoderModel.from_pretrained(MODEL_NAME).eval().to(DEVICE)
TPB = 16000 if DEVICE == "cuda" else 8000
log("encoder", MODEL_NAME, "on", DEVICE, "| fp32")

def enc(texts, tokens_per_batch=TPB):
    order = sorted(range(len(texts)), key=lambda i: len(texts[i])); res = np.zeros((len(texts), model.config.d_model), np.float32); s0 = 0
    while s0 < len(order):
        e = s0 + 1
        while e < len(order) and e - s0 < 64 and (e - s0 + 1) * min(MAXTOK, len(texts[order[e]]) // 3 + 4) <= tokens_per_batch: e += 1
        b = order[s0:e]; s0 = e
        t = tok([texts[i] for i in b], max_length=MAXTOK, truncation=True, padding=True, return_tensors="pt").to(DEVICE)
        with torch.inference_mode():
            h = model(**t).last_hidden_state; m = t["attention_mask"][:, :, None].to(h.dtype)
            res[b] = ((h * m).sum(1) / m.sum(1)).float().cpu().numpy()
    return res

FEATS, SPLIT_INFO = {}, {}      # key -> {"cal","fit","test"} raw float32 features
os.makedirs(os.path.join(WORK, "feats_" + MODEL_NAME), exist_ok=True)

def cached(key, make):
    f = os.path.join(WORK, "feats_" + MODEL_NAME, key.replace("/", "__") + (".quick" if QUICK else "") + ".npz")
    if os.path.exists(f):
        z = np.load(f); return {k: z[k] for k in z.files}
    d = make(); np.savez(f, **d); return d

def bench_task(folder, name):
    tr, te = load_inputs(folder, name, "train"), load_inputs(folder, name, "test")
    if QUICK: tr, te = tr[:40], te[:20]
    n = len(tr); perm = list(range(n)); random.Random(0).shuffle(perm); sel = perm[:min(n, 400)]; m = 200 if n >= 400 else n // 2
    X = enc([tr[i] for i in sel]); SPLIT_INFO[f"{folder}/{name}"] = {"n_train": n, "m_heldout": m, "n_fit": len(sel) - m, "n_test": len(te)}
    return {"cal": X[:m], "fit": X[m:], "test": enc(te)}

def random_split():
    """The second version of the fresh split: ONE seed-0 shuffle of all 1,000 instances; held-out = first 200, fit = next 200, test = next 200."""
    perm = list(range(1000)); random.Random(0).shuffle(perm)
    return perm[:200], perm[200:400], perm[400:600]

def fresh_task(name):
    """Features of the union of the positions used by either split, computed once; both splits index into them."""
    x = FRESH_INPUTS[name]; n = len(x); sp = [list(a) for a in fresh_split()]; rp = [list(a) for a in random_split()]
    sp = [[j for j in a if j < n] for a in sp]; rp = [[j for j in a if j < n] for a in rp]
    if QUICK: sp = [a[:20] for a in sp]; rp = [a[:20] for a in rp]
    pos = sorted(set(sum(sp, []) + sum(rp, []))); X = enc([x[j] for j in pos]); at = {j: i for i, j in enumerate(pos)}
    SPLIT_INFO["fresh/" + name] = {"n_instances": n, "union_positions": len(pos), "official_like": [len(a) for a in sp], "random": [len(a) for a in rp]}
    out = {}
    for pre, parts in (("", sp), ("r", rp)):
        for k, a in zip(("cal", "fit", "test"), parts): out[pre + k] = X[[at[j] for j in a]]
    return out

bench_keys = [("SuperNI", n) for n in ORDER1] + [("Long_Sequence", n) for n in LONG3]
for folder, name in bench_keys:
    FEATS[f"{folder}/{name}"] = cached(f"{folder}/{name}", lambda: bench_task(folder, name))
    log("features", f"{folder}/{name}", {k: v.shape for k, v in FEATS[f"{folder}/{name}"].items()})
for i, (name, _, _) in enumerate(FRESH):
    if name not in FRESH_INPUTS: continue
    FEATS["fresh/" + name] = cached("fresh/" + name, lambda: fresh_task(name))
    if i % 10 == 9: log("features of fresh tasks up to", i + 1)
for k in list(FEATS):
    if k not in SPLIT_INFO: SPLIT_INFO[k] = {"from_cache": True, "shapes": {a: list(b.shape) for a, b in FEATS[k].items()}}
bad = [k for k, v in FEATS.items() if min(len(v[k]) for k in v) == 0]
if bad: print("WARNING: tasks with an empty split, left out:", bad); [FEATS.pop(k) for k in bad]
del model
if DEVICE == "cuda": torch.cuda.empty_cache()
if QUICK:       # the pool statistics are for t5-base (768 numbers); with t5-small the statistics come from the fit features themselves
    ST = standardiser(np.vstack([v["fit"] for v in FEATS.values()]))
else:
    ST = standardiser_embedded(CONST["mu"], CONST["sd"])
log("features done:", len(FEATS), "tasks;", sum(len(v["test"]) for v in FEATS.values()), "test inputs;", sum(len(v["fit"]) + len(v["cal"]) for v in FEATS.values()), "fit+held-out inputs")

## Sequences and rules

In [ ]:
FRESH_KEYS = ["fresh/" + n for n, _, _ in FRESH if "fresh/" + n in FEATS]
S1 = ["SuperNI/" + n for n in ORDER1]; S2 = ["SuperNI/" + n for n in ORDER2]
L3 = ["Long_Sequence/" + n for n in LONG3]; L4 = ["Long_Sequence/" + n for n in LONG4]
fk = lambda a, b: [k for k in FRESH_KEYS if a <= [n for n, _, _ in FRESH].index(k[6:]) < b]      # tasks of the batches, in the order of the batches
SEQS = {"superni_order1": S1, "superni_order2": S2, "long_order3": L3, "long_order4": L4,
        "fresh30": fk(0, 30), "fresh30_random": fk(0, 30), "fresh60": fk(0, 60), "fresh60_random": fk(0, 60),
        "fresh90": fk(0, 90), "fresh90_random": fk(0, 90), "all120": S1 + L3 + fk(0, 90), "all120_random": S1 + L3 + fk(0, 90)}
# "_random": the fresh tasks use the second split (one random split of all 1,000 instances); benchmark tasks keep their official test inputs
PAIRS = [("fresh30", "fresh30_random"), ("fresh60", "fresh60_random"), ("fresh90", "fresh90_random"), ("all120", "all120_random")]
J_MAX_TASKS = 30
RESULTS = {"about": "Hidden states of the frozen T5 encoder at scale (PLAN.md 2026-11-18 07:48 UTC). Rules L (ridge lam 1), H' (t = 2, floor) and J (C = 1) of hidden_rules_study.py.",
           "quick": QUICK, "model": MODEL_NAME, "device": DEVICE, "official_commit": OFFICIAL_COMMIT, "natural_instructions_commit": NI_COMMIT, "max_tokens": MAXTOK,
           "pool_sha1": CONST["pool_sha1"], "pool_shape": CONST["pool_shape"], "standardised_with": "the embedded pool statistics" if not QUICK else "fit features (QUICK only)",
           "fingerprint_check": {"matching": MATCH, "of": len(FRESH), "data_differs": DATA_DIFFERS, "download_failed": DOWNLOAD_FAILED},
           "long_order4": LONG4, "split_info": SPLIT_INFO, "sequences": {}, "timings_s": {}}
RES_FILE = os.path.join(OUT_DIR, "results.json")
def dump():
    json.dump(RESULTS, open(RES_FILE + ".tmp", "w")); os.replace(RES_FILE + ".tmp", RES_FILE)

def pack(r):
    return {"mean": r["mean"], "se": r["se"], "per_task": r["per_task"], "certificate_valid": r["certificate_valid"]}

def arrays(keys, tag):
    pre = lambda k: "r" if (tag.endswith("_random") and k.startswith("fresh/")) else ""
    return [ST(FEATS[k][pre(k) + "fit"]) for k in keys], [ST(FEATS[k][pre(k) + "cal"]) for k in keys], [ST(FEATS[k][pre(k) + "test"]) for k in keys]

def run_LH(tag):
    keys = SEQS[tag]; t = time.time(); fit, cal, te = arrays(keys, tag)
    Rt, Rc, _ = routes_ridge(fit, cal, te, 1); L = measures(keys, Rt, Rc); del Rt, Rc
    Rt, Rc, ex = routes_handicap(fit, cal, te, 1, [2], floor=True)[2]; H = measures(keys, Rt, Rc, extra=ex, held_out_used_by_rule=True, handicap_t=2); del Rt, Rc
    res = RESULTS["sequences"].setdefault(tag, {"tasks": keys}); res["L"] = pack(L); res["H_prime_t2"] = pack(H)
    if tag in ("long_order3", "long_order4"):
        m, s = L["mean"], L["se"]
        res["reading_rule_L"] = {"own_at_end": m["own_at_end"],
                                 "use": "works on the second benchmark" if m["own_at_end"] >= 0.90 else ("does not" if m["own_at_end"] < 0.75 else "partial"),
                                 "mean_changed": m["changed"], "mean_certificate": m["certificate"], "se_changed": s["changed"],
                                 "audit": "met" if m["changed"] <= m["certificate"] + 2 * s["changed"] else "not met"}
    RESULTS["timings_s"][tag + "|L,H"] = round(time.time() - t, 1); dump(); log(tag, "L and H' done", round(time.time() - t), "s")

def run_J(tag):
    keys = SEQS[tag]; t = time.time(); fit, cal, te = arrays(keys, tag)
    Rt, Rc = routes_joint(fit, cal, te, 1); J = measures(keys, Rt, Rc); del Rt, Rc
    RESULTS["sequences"][tag]["J"] = pack(J); RESULTS["timings_s"][tag + "|J"] = round(time.time() - t, 1); dump(); log(tag, "J done", round(time.time() - t), "s")

for tag in SEQS: run_LH(tag)
RESULTS["fresh_split_comparison"] = {"about": "random split minus official-like split (fresh_split), difference of the means over tasks"}
for a, b in PAIRS:
    RESULTS["fresh_split_comparison"][b + " - " + a] = {rule: {k: RESULTS["sequences"][b][rule]["mean"][k] - RESULTS["sequences"][a][rule]["mean"][k] for k in ("changed", "changed_ever", "own_at_end")}
                                                         for rule in ("L", "H_prime_t2")}
dump()

## Rule J (joint logistic regression, C = 1) for the sequences of at most 30 tasks
Retrained at every arrival with scikit-learn; for the larger sequences (60, 90, 120 tasks) it is skipped (the time would be too long) and the result says so.

In [ ]:
J_BUDGET_S = 2.5 * 3600 if not QUICK else 600; tJ = time.time()
for tag in SEQS:
    if len(SEQS[tag]) > J_MAX_TASKS:
        RESULTS["sequences"][tag]["J"] = f"skipped: {len(SEQS[tag])} tasks (rule J is run only for sequences of at most {J_MAX_TASKS} tasks)"; continue
    if time.time() - tJ > J_BUDGET_S:
        RESULTS["sequences"][tag]["J"] = "skipped: time budget of rule J used up"; continue
    run_J(tag)
dump()

## Tables

In [ ]:
KEYS = ["own_at_arrival", "own_at_end", "changed", "changed_ever", "certificate", "upper95", "prior_bound"]
def line(name, r, extra=()):
    return f"  {name:6s}" + "".join(f" {r['mean'][k]:.4f}+-{r['se'][k]:.4f}" for k in KEYS + list(extra))
for tag, res in RESULTS["sequences"].items():
    print(f"\n== {tag}: {len(res['tasks'])} tasks (mean +- standard error over tasks)")
    print("        " + "".join(f" {k[:15]:>15s}" for k in KEYS) + "   (H': handicap_bound, takes_of_own_heldout)")
    print(line("L", res["L"])); print(line("H'", res["H_prime_t2"], ("handicap_bound", "takes_of_own_heldout")))
    print(line("J", res["J"]) if isinstance(res.get("J"), dict) else "  J      " + str(res.get("J")))
    if "reading_rule_L" in res: print("  reading of rule L:", res["reading_rule_L"]["use"], "| audit", res["reading_rule_L"]["audit"], "| own_at_end", round(res["reading_rule_L"]["own_at_end"], 4))
print("\nShare given to the own task after the last task, against the number of tasks (rule L / H' / J):")
for tag in ("superni_order1", "long_order3", "fresh30", "fresh30_random", "fresh60", "fresh60_random", "fresh90", "fresh90_random", "all120", "all120_random"):
    r = RESULTS["sequences"][tag]; J = r.get("J")
    print(f"  {tag:15s} {len(r['tasks']):4d} tasks  L {r['L']['mean']['own_at_end']:.4f}  H' {r['H_prime_t2']['mean']['own_at_end']:.4f}  J " + (f"{J['mean']['own_at_end']:.4f}" if isinstance(J, dict) else "-"))
print("\nRandom split minus fresh_split (rule L):")
for a, b in PAIRS: print(f"  {b} - {a}: " + "  ".join(f"{k} {v:+.4f}" for k, v in RESULTS["fresh_split_comparison"][b + " - " + a]["L"].items()))
for tag in ("long_order3", "long_order4", "superni_order1"):
    res = RESULTS["sequences"][tag]
    print(f"\nPer task, {tag}, rule L / H' (arrival, end, changed, certificate (c/m), top wrong destination):")
    for k in res["tasks"]:
        a, b = res["L"]["per_task"][k], res["H_prime_t2"]["per_task"][k]
        print(f"  {k[:34]:34s} L {a['own_at_arrival']:.3f} {a['own_at_end']:.3f} {a['changed']:.3f} {a['certificate']:.3f} ({a['c']}/{a['m']}) -> {str(a.get('top_wrong_destination', ''))[:22]:22s}"
              f" | H' {b['own_at_arrival']:.3f} {b['own_at_end']:.3f} {b['changed']:.3f} {b['certificate']:.3f}")
RESULTS["timings_s"]["total"] = round(time.time() - T0, 1); dump()
print("\nresults written to", RES_FILE, "| total", round(time.time() - T0), "s")